# データ・AI活用実践（初級）第05回

-------------------

## 前回の状態の復元

本日は前回の続きといえます．したがって，とりあえず前回の最後の状態を復元しないことには何事も進みません．したがって，`pref47_temp.csv`および`JIS.csv`いうファイルを取得してください．以下のソースコードでダウンロードしてもいいし，自分で作成した同名のファイルをアップロードしてもいいです．

取得できたら，`pref47_temp.csv`を読み込みます．まずはライブラリです．

In [ ]:
# ここは覚えなくても大丈夫！
csvurl <- "https://www.cc.kyoto-su.ac.jp/~ogohara/lecture/DataAI_FirstCourse/pref47_temp.csv"
download.file(csvurl, destfile = "pref47_temp.csv") #pref47_temp.csvという名前のファイルとしてダウンロード

In [ ]:
# ここは覚えなくても大丈夫！
csvurl <- "https://www.cc.kyoto-su.ac.jp/~ogohara/lecture/DataAI_FirstCourse/JIS.csv"
download.file(csvurl, destfile = "JIS.csv") #JIS.csvという名前のファイルとしてダウンロード

In [ ]:
library(tidyverse)

In [ ]:
df<-read_csv('pref47_temp.csv', locale=locale(encoding='Shift_JIS'))

前回と同様さまざまなライブラリを利用します．「前回インストールしたやん」とお思いかもしれませんが，自分でインストールしたライブラリは寝て覚めたら消去された状態になります．したがって，やるたびに再インストールする必要があるのです．例によって時間がかかるので，最初にどかっとインストールしておきましょう．

In [ ]:
system("apt-get install -y libudunits2-dev libgdal-dev libgeos-dev libproj-dev") #もしかしたらいらないかもしれない
install.packages('sf')

In [ ]:
install.packages('NipponMap')

In [ ]:
system('apt-get install libprotobuf-dev protobuf-compiler')
system('apt-get install libjq-dev')
install.packages('geojsonio')

In [ ]:
library(sf)
library(NipponMap)

データフレーム`df`の中身をここで確認しましょう．

In [ ]:
df

余計な1列が一番左に入っているかもしれません．まああってもいいのですが，せっかく苦労して行った前処理が無駄になっている気がして癪なので消してしましましょう．もちろん，csvファイルを保存するときに，こういう無駄な列が入らないように保存する方法はありますので，我慢ならない人は調べてみてください．

In [ ]:
# 整数列からなる余計な1列が左端に入っていない人は
# df47 <- df
# としてください．
df47 <- df[,-1]

では，一気に前回の最終状態まで行きます．説明のためにコードが分散していましたが，ここでは一つにまとめてしまいます．1行1行何をやっているか理解できていれば御の字です．

In [ ]:
# まずは４７都道府県の気温データを整えます
mtsum <- apply(df47[,-1], 2, summary)
dfsum <- data.frame(t(mtsum))
# shape file読み込み
map <- read_sf(system.file("shapes/jpn.shp", package = "NipponMap")[1],
                crs = "+proj=longlat +datum=WGS84")
# JISデータ読み込み
jisdf <- read_csv('JIS.csv', locale=locale(encoding='Shift_JIS'))

In [ ]:
# 前回と同じコードです．復習しましょう！
city <- c('那覇', '松江', '松山', '高松', '神戸', '津', '彦根', '金沢', '名古屋', '前橋', '甲府', '横浜', '熊谷', '宇都宮', 'つくば（館野）', '仙台', '盛岡', '札幌')
pref <- c('沖縄', '島根', '愛媛', '香川', '兵庫', '三重', '滋賀', '石川', '愛知', '群馬', '山梨', '神奈川', '埼玉', '栃木', '茨城', '宮城', '岩手', '北海道')

for (n in jisdf$都道府県名){
  nlen <- str_length(n)
  if (n=="北海道") {
    prefname <- n
  } else {
    prefname <- str_sub(n, end=nlen-1)
  }

  if ( is.na(match(prefname,pref)) ) {
    name_in_dfsum <- prefname
  } else {
    name_in_dfsum <- city[which(pref == prefname)]
  }
  dfsum[name_in_dfsum,'jiscode'] <- jisdf[ which(jisdf$都道府県名 == n),"都道府県コード" ]
}

map[,'temperature'] <- dfsum[order(dfsum$jiscode),'Mean']
map

In [ ]:
# 最後に描画しましょう．
library(RColorBrewer)
ggplot(map, aes(fill = temperature)) + geom_sf() + labs(title = "Mean Temperature") + scale_fill_gradientn(colours=topo.colors(9))

できましたか？なんだかいろいろやった気がしますが，改めてみればたったこれだけでした．

--------------------
## 気候区分

さあ，文系の方，出番です．中学や高校の地理の授業で，「気候区分」を習ったと思います．実を言いますと，気候の区分方法は無数にあります．そのうち最も有名で，皆さんも一度は聞いたことがある分類方法はケッペンの気候区分です．ただしそれも，５区分だったり，６区分だったり，１３区分だったりするのですが，まあここではいいでしょう．とりあえず，寒帯とか，熱帯とか，温帯とか，聞いたことがあると思います．ケッペンの気候区分は，植生，気温，降水量などを基に決められます．さすがにケッペンの気候区分をもう一回やってみるのはしんどいので，本日はとても雑に日本の都道府県を複数の気候に分けてみましょう．

### 平均気温を用いた簡単な閾値処理

例えば，日本の都道府県を3つの気候帯に分けなさい，と言われて，皆さんはどうするでしょうか．北海道，沖縄，それ以外，とかでしょうか？一番簡単な方法は，境目となる気温を適当に2つ決めて，寒いところ，熱いところ，その中間，とすることですね．2つの気候帯に分けなさい，と言われれば，閾値となる気温1つを決めればよくなります．都道府県の平均気温という**特徴**1つだけを使うのであれば，結局のところ複数の閾値処理で，えいっと2つや3つや4つにわけることになりそうです．では，この時の閾値，境目となる気温の値をどう決めましょうか．そのためにはまず，ヒストグラムを書いてみましょう．

In [ ]:
g <- ggplot(dfsum, aes(x=Mean) )
g <- g+geom_histogram()
plot(g)

ま，まあかけました．地味ですけど．このヒストグラムを見て，ここに境目を作ればうまく3つに分けられそう！，と思える平均気温はあるでしょうか？例えば14度と20度くらいでしょうか．では平均気温が14度よりも低い都道府県はどこでしょうか．

In [ ]:
dfsum[ dfsum$Mean<14,]

`dfsum`の`Mean`列が14未満の行だけ抜き出しました．

結果はこんな感じです．皆さんの直感とそれほど外れていないと思います．平均気温が20度より高いのはおそらく沖縄でしょうから，さすがにそれは確認しなくていいでしょう．ではここで想像してください．47都道府県ではなく，もしすべての市町村の平均気温データがあったらどうなるでしょうか．基本的には上のようなヒストグラムになりますが，ヒストグラムはもっと密になり縦軸の度数の値はずっと大きくなります．その場合，14度付近に見つけやすい境目があるとは限らず，そのあたりのヒストグラムはなめらかかもしれません．また，たとえ47都道府県の場合でも，閾値が14度ではなくて10度ではだめなのでしょうか．北海道と東北を同じものにするか，北海道だけ寒い地域とするか，ヒストグラムからは自明ではありません．

### 最低気温と最高気温の散布図

`dfsum`には平均気温以外にも最低気温や最高気温データがあります．平均気温だけではなくて，ほかの量を使用することでもっと客観的に分けることはできないでしょうか．2変数であれば，散布図を作成することで全体的な印象をつかむことができます．

In [ ]:
g <- ggplot(dfsum, aes(x=Min., y=Max., fill=Mean) ) #dfsumのうち，Min.をx軸，Max.をy軸にして，色はMeanに基づいて決めて．
g <- g+geom_point(shape=21, size=2)                 #21番のマーカーを使って，sizeは2 pointの点々を書いてね．
g <- g+scale_fill_gradientn(colours=topo.colors(9)) #topoというcolor mapに従って，9色で塗り分けて
plot(g) #描け！

In [ ]:
dfsum[ dfsum$Mean<14,]

どうでしょう？散布図の点々の形`shape`と大きさ`size`を変更しました．点々の色をデータフレームの`Mean`に従って連続的に変化するように変更しまし，色合いを日本地図の都道府県と同じにしました（`scale_fill_gradientn(colours=topo.colors(9))`）．

右端の沖縄は依然として一人ぽつんと別気候帯であることはよくわかります．最低気温が15度くらいにぶっ飛んでいるからです．寒い側の一団は，左下の7点が何となく他と分けられている印象です．一番左下は札幌なので，散布図で見れば，札幌の右上で区分けするよりかは，仙台や山形の右上で区切ったほうがしっくりきませんか？（仙台や山形がどの点なのか，データフレームの中身と比べながら確認してください．）本来ならば，最低気温や最高気温，平均気温に加えて，降水量なども加味して気候区分を行うのが正しいですが，これ以上は踏み込みません．大事なことは，2次元の散布図（場合によってはより高次元の散布図）を1つ確認するだけで，「ざっくり」と自然に分かれている場合があることです．だったら，「勝手に機械が判断してくれへんかな．．．．」と思いませんか？私だったら思います．

### クラスタリング

実は勝手にやったもらうことができます．**教師なし学習**といわれる分野です．教師がいないのに何を習うのか，とお思いかもしれませんが，私もそう思います．もちろん，あらゆる場合に信頼できるわけではなく，その結果は自分で確認して良し悪しを決めなければなりませんが，現代ではあまりにも簡単に試してみることが可能です．

In [ ]:
set.seed(1000)  #　呪文
km1 <- kmeans(dfsum[,c('Min.','Max.')],3)
km1

終わりました．`Min.`と`Max.`を用いてデータを3つにクラス分けしたのです．3クラス分類ですね．kmeans法と呼ばれるアルゴリズムを用いて，データの特徴のみから，データを3つに分けました．この方法は，分類すべきクラス数がわかっている場合のみ使うことができるのですが，それにしても簡単すぎます．上の出力結果では，

- 3クラスに分類し，それぞれ23，18，6の都道府県から成る
- 3つのクラスに1, 2, 3の番号をつければ，クラス1の最低気温の平均値は1.943478度，最高気温の平均は29.42174度，クラス2のそれらは．．．．となる．
- 各都道府県のクラスはこの通り
- `km1`という返り値にはいろいろ情報を入れてあります．

> **注：これら3つのクラスの番号が入れ替わっている可能性はあります．しかし，よく見れば順番が異なるだけだとわかるでしょう．**

ということがわかります．`km1`という返り値の`cluster`というcomponentにはクラス番号が入っています．

In [ ]:
km1$cluster

ですので，`dfsum`に結合してしまいましょう．

In [ ]:
dfsum$cluster <- km1$cluster
dfsum

ということは，今まで通り散布図に色を付けることが可能ですね！

In [ ]:
g <- ggplot(dfsum, aes(x=Min., y=Max., fill=cluster) ) #色はMeanじゃなくてclusterに基づいて決めて
g <- g+geom_point(size=2, shape=21)
g <- g+scale_fill_gradientn(colours=topo.colors(3))
plot(g)

あれれ，なんだか想定と違います．札幌だけ仲間外れになっていません．確かに！真ん中のクラスと右のクラスの間には断絶があります．．．．ここを分けるのか．．．という具合に，機械の判断を人間の判断が同じとは限りません．実を言いますと，kmeans法のアルゴリズム上，縦軸と横軸のスケールを同じにしたほうがこちらの意図に沿うことが多いです．いま横軸は-5から+15なので20度くらいの幅がありますが，縦軸は5-6度くらいの幅しかありません．つまり上の図は実は横に間延びしているのです．だから横軸方向は見た目以上にスカスカで，横方向に3分割したほうが**おトク**と思っているのです．こういう場合は，縦軸と横軸の値を標準化してしまいます．

In [ ]:
dfsum[,c('Minn','Maxn')] <- scale( dfsum[,c('Min.','Max.')])
dfsum

こうすると，平均0，分散1になります．新しい列`Minn`や`Maxn`は0に近い数値で正だったり負だったりしますよね．この新しい列の数値を使って改めてクラスタリングしてみましょう．

In [ ]:
set.seed(1000) #呪文
km2 <- kmeans(dfsum[,c('Minn','Maxn')],3) #3クラス分類
dfsum$clusnor <- km2$cluster              #分類結果を新しい列にする
#g <- ggplot(dfsum, aes(x=Minn, y=Maxn, fill=clusnor) ) #縦軸横軸は標準化された値．色はclusnorに基づいて決めて
g <- ggplot(dfsum, aes(x=Min., y=Max., fill=clusnor) ) #縦軸横軸は標準化前の値．色はclusnorに基づいて決めて
g <- g+geom_point(size=2, shape=21)
g <- g+scale_fill_gradientn(colours=topo.colors(3))
plot(g)

う，うーん．．．．まだかーーーーー！一見うまくいったように見えますが，やはり沖縄とその他の間で別れません．寒い地域7県はばっちり分かれてますね．横軸だけで分類せずに，斜めに分けている感じが出ています．

機械の肩を持つわけではありませんが，これはこれでいいのです．機械が勝手なことをしているわけではありません．そうしろ，と人間が指示しているからそうしているだけです．したがって，この結果が不満ならば，悪いのは我々であって機械ではありません．我々がやりたいことと，我々が指示していることがちぐはぐなのです．よくよく見れば，機械のクラスタリングだって一理あるように見えませんか？

最後に3つの気候区分で色分けした日本地図を書いておきましょう．

In [ ]:
map[,'normclus'] <- dfsum[order(dfsum$jiscode),'clusnor']
ggplot(map, aes(fill = normclus)) + geom_sf() + labs(title = "都道府県別の簡単気候区分") + scale_fill_gradientn(colours=topo.colors(9))

おもしろいですね！寒くもなく暖かくもない（逆に言えば，寒くもあり暖かくもある）地域は，

- 山陰
- 北陸
- 北関東
- 南東北
- **内陸**

ですね．長野が寒い地域判定されているので，内陸は一段階寒く判定されるのでしょうか？

本来なら，もうこの段階で仕事の80%くらいは終わっているのですが，この散布図に「新しいデータ」が追加される場合に，それがどの区分に所属するか決めたい場合があります．例えば，東京都を23区とそれ以外と，小笠原に分けて考えたい場合などです．そうすると，暖かい地域，寒い地域，マイルドな地域の境界線が具体的にわかっていないと，次に来る新しいデータがどこの区分に入るのかわかりません．

-----------------
## 自由課題

1. 上のソースコードをコピーして，クラス数を4にしてクラスタリングしてみましょう．4色に色分けして散布図を描画してください．
1. この結果を日本地図で描画してください．
1. 2クラス分類や10クラス分類で同じことをどんどんやってみよう！



------------
# 採点対象の課題

回答はmoodleの小テスト/課題から送信してください．

1. 寒暖の差（最高気温-最低気温）が最も大きい都道府県を答えてください．
1. 平均気温が，長野県だけ周囲と比べて低い理由を説明してください．
1. 沖縄県とその他の間にクラス境界が存在するようにするために試すとよいことを考えてください．（沖縄県とその他の間のクラス境界以外のクラス境界はどうなってもいいです．）
1. この演習の最後に取り組んだ（標準化した）3クラス分類の結果を使います．新たに最低気温`Minn`=1，最高気温`Maxn`=-2.5のデータが手に入りました．どのクラスに分類されるべきですか？（ヒント：クラス中心までの距離が最も短いクラスに分類されます．）
